# Noon-shift test (option #4)

**Date:** 2026-04-18.
**Hypothesis:** Shifting day-level review timestamps from midnight UTC to noon UTC eliminates the "boundary-on-spike" artifact that made midnight-aligned snap under-predict cohort-wide.

**Mechanism:** Day-level reviews default to midnight UTC of their day. Under midnight-aligned snap (00:00 UTC on close-3 → dbc=3.583), training's day-level reviews on close-3 at midnight UTC have dbc=3.583 — EXACTLY on the phase-1 upper boundary. KDE integration captures only half the spike's smoothed mass.

Noon-shift moves day-level reviews to 12:00 UTC of their day → dbc=3.083. Now they're cleanly inside phase-1 window `(0.583, 3.583]`. No boundary-on-spike.

**Note:** Noon shift only affects reviews with `timestamp_confidence == 'd'`. H/m reviews retain their precise timestamps.

**Test grid (2×2):**
| snap \ shift | no_shift | noon_shift |
|---|---|---|
| original (14:00 UTC close-3) | baseline (ship) | should be ~identical (spike was already off-boundary) |
| midnight (00:00 UTC close-3) | prior test — cohort under-predicted | **candidate — should fix it** |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

import _helpers  # we'll swap the reviews module variable
from _helpers import (
    close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP_DAYS = 3

CACHE = CACHE_DIR / 'noon_shift_test.pkl'

# Build two review DataFrames: original, and noon-shifted
from _helpers import reviews as reviews_orig
reviews_shifted = reviews_orig.copy()
day_mask = reviews_shifted['timestamp_confidence'] == 'd'
n_shifted = day_mask.sum()
reviews_shifted.loc[day_mask, 'estimated_timestamp'] = (
    reviews_shifted.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)
print(f'Noon-shift: moved {n_shifted} day-level reviews (+12h), left {(~day_mask).sum()} h/m reviews unchanged')

## Generalized helpers

Rebuild snapshot state, actuals, and first_review_ts from whichever reviews frame we use.

In [ ]:
def snapshot_state_g(reviews_df, target_slug, snap_time, target_close):
    obs = reviews_df[
        (reviews_df['movie_slug'] == target_slug)
        & (reviews_df['estimated_timestamp'] < snap_time)
        & (reviews_df['estimated_timestamp'] < target_close)
    ]
    if obs.empty:
        return None
    return {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float(
            (target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400
        ),
    }

def actual_phase1_g(reviews_df, target_slug, snap_time, target_close):
    """Count reviews in [snap_time, midnight UTC of close_day)."""
    close_midnight = target_close.floor('D')
    mr = reviews_df[reviews_df['movie_slug'] == target_slug]
    in_window = (
        (mr['estimated_timestamp'] >= snap_time)
        & (mr['estimated_timestamp'] < close_midnight)
    )
    return int(in_window.sum())

def critics_in_window_g(reviews_df, slug, window_start, window_days):
    window_end = window_start + pd.Timedelta(days=window_days)
    mr = reviews_df[
        (reviews_df['movie_slug'] == slug)
        & (reviews_df['estimated_timestamp'] >= window_start)
        & (reviews_df['estimated_timestamp'] <= window_end)
    ]
    return set(mr['reviewer_name'])

def jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return 0.0
    return len(set_a & set_b) / len(union)

def first_review_ts_g(reviews_df):
    return (reviews_df[reviews_df['movie_slug'].isin(close_date_map)]
            .groupby('movie_slug')['estimated_timestamp'].min())

def combined_score_g(reviews_df, first_rev_ts, target, target_gap, target_critics,
                     target_window_days, k, alpha, sigma_gap):
    target_close = close_date_map[target]
    from _helpers import gaps
    candidates = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target)
    ]
    rows = []
    for _, row in candidates.iterrows():
        slug = row['slug']
        gap_diff = abs(row['gap_days'] - target_gap)
        gap_score = float(np.exp(-gap_diff / sigma_gap)) if not np.isinf(sigma_gap) else 1.0
        if slug not in first_rev_ts.index:
            continue
        train_first = first_rev_ts.loc[slug]
        train_critics = critics_in_window_g(reviews_df, slug, train_first, target_window_days)
        j = jaccard(target_critics, train_critics)
        combined = alpha * gap_score + (1 - alpha) * j
        rows.append((slug, combined))
    rows.sort(key=lambda x: x[1], reverse=True)
    return {slug: score for slug, score in rows[:k]}

def build_profiles_g(reviews_df, training_scores):
    training_slugs = list(training_scores.keys())
    n_movies = len(training_slugs)
    raw_weights = np.array([training_scores[s] for s in training_slugs], dtype=float)
    total_w = raw_weights.sum()
    if total_w <= 0:
        norm_weights = np.ones_like(raw_weights)
    else:
        norm_weights = raw_weights * (n_movies / total_w)
    slug_weight = dict(zip(training_slugs, norm_weights))

    train = reviews_df[reviews_df['movie_slug'].isin(training_slugs)].copy()
    close_map_series = pd.Series(close_date_map)
    train['bet_close'] = train['movie_slug'].map(close_map_series)
    train['days_before_close'] = (
        train['bet_close'] - train['estimated_timestamp']
    ).dt.total_seconds() / 86400
    train = train[train['days_before_close'] > 0].copy()
    train['movie_weight'] = train['movie_slug'].map(slug_weight)

    rows = []
    for name, group in train.groupby('reviewer_name'):
        movies_seen = group['movie_slug'].unique()
        base_rate = float(sum(slug_weight[s] for s in movies_seen) / n_movies)
        fresh = (group['tomatometer_sentiment'] == 'positive').sum()
        total = len(group)
        timing = group['days_before_close'].values.tolist()
        weights = group['movie_weight'].values.tolist()
        rows.append({
            'reviewer_name': name, 'base_rate': base_rate,
            'fresh_rate': fresh / total if total > 0 else 0.5,
            'timing_data': timing, 'timing_weights': weights, 'n_reviews': total,
        })
    df = pd.DataFrame(rows, columns=['reviewer_name','base_rate','fresh_rate',
                                      'timing_data','timing_weights','n_reviews'])
    from rotten_tomatoes_forecasting.critic_model import CriticProfiles
    return CriticProfiles(df=df, training_slug_count=n_movies)


# Pre-compute first_review_ts for both frames
first_rev_orig = first_review_ts_g(reviews_orig)
first_rev_shifted = first_review_ts_g(reviews_shifted)
print(f'first_review_ts: {len(first_rev_orig)} orig, {len(first_rev_shifted)} shifted')

## Run the 2×2 sweep

In [ ]:
CONFIGS = [
    ('orig_snap__no_shift',      False, False),
    ('orig_snap__noon_shift',    False, True),
    ('midnight_snap__no_shift',  True,  False),
    ('midnight_snap__noon_shift', True, True),
]

def get_snap_time(close_ts, midnight_aligned):
    if midnight_aligned:
        return close_ts.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    return close_ts - pd.Timedelta(days=SNAP_DAYS)

def run_sweep(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    for target in close_date_map:
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        row = {'target': target, 'target_gap': target_gap}

        for cfg_name, midnight_aligned, noon_shifted in CONFIGS:
            revs = reviews_shifted if noon_shifted else reviews_orig
            first_rev = first_rev_shifted if noon_shifted else first_rev_orig
            snap_time = get_snap_time(target_close, midnight_aligned)
            snap_dbc_effective = (target_close - snap_time).total_seconds() / 86400

            state = snapshot_state_g(revs, target, snap_time, target_close)
            if state is None:
                row[f'{cfg_name}_pred'] = np.nan
                row[f'{cfg_name}_actual'] = np.nan
                continue
            if state['first_review_dbc'] < snap_dbc_effective + 1.0:
                row[f'{cfg_name}_pred'] = np.nan
                row[f'{cfg_name}_actual'] = np.nan
                continue
            if len(state['observed_critics']) < 3:
                row[f'{cfg_name}_pred'] = np.nan
                row[f'{cfg_name}_actual'] = np.nan
                continue

            target_window_days = state['first_review_dbc'] - snap_dbc_effective
            scores = combined_score_g(
                revs, first_rev, target, target_gap,
                state['observed_critics'], target_window_days,
                k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
            )
            if len(scores) < 5:
                row[f'{cfg_name}_pred'] = np.nan
                row[f'{cfg_name}_actual'] = np.nan
                continue

            try:
                profiles = build_profiles_g(revs, scores)
                if len(profiles.df) == 0:
                    raise ValueError('empty profiles')
                model = build_weighted_kde_lambda_model(
                    profiles,
                    bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
                    bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
                )
                pred = predict_window_custom(
                    model, dbc_from=snap_dbc_effective, dbc_to=midnight_utc_dbc,
                    observed_critics=state['observed_critics'],
                    observed_count=state['observed_count'],
                    first_review_dbc=state['first_review_dbc'],
                )
            except Exception:
                row[f'{cfg_name}_pred'] = np.nan
                row[f'{cfg_name}_actual'] = np.nan
                continue

            actual_p1 = actual_phase1_g(revs, target, snap_time, target_close)
            row[f'{cfg_name}_pred'] = float(pred)
            row[f'{cfg_name}_actual'] = actual_p1

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

results = run_sweep()
# Keep only rows where all 4 configs have valid predictions
valid = results.dropna(subset=[f'{c[0]}_pred' for c in CONFIGS])
print(f'Targets with all 4 configs valid: {len(valid)}')

## Cohort-wide MAE comparison

In [ ]:
print('Cohort MAE by config:\n')
print(f'  {"config":35s}  {"MAE":>7s}  {"mean_err":>10s}')
for cfg_name, _, _ in CONFIGS:
    errs = valid[f'{cfg_name}_pred'] - valid[f'{cfg_name}_actual']
    mae = errs.abs().mean()
    me = errs.mean()
    print(f'  {cfg_name:35s}  {mae:7.2f}  {me:+10.2f}')

print()
# Quartile by orig-snap actual_phase1
valid = valid.copy()
valid['q_actual'] = pd.qcut(valid['orig_snap__no_shift_actual'], q=4,
                             labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

for q in ['Q1','Q2','Q3','Q4']:
    sub = valid[valid['q_actual'] == q]
    if not len(sub):
        continue
    lo = int(sub['orig_snap__no_shift_actual'].min())
    hi = int(sub['orig_snap__no_shift_actual'].max())
    print(f'{q} (orig actual [{lo},{hi}]) n={len(sub)}:')
    for cfg_name, _, _ in CONFIGS:
        errs = sub[f'{cfg_name}_pred'] - sub[f'{cfg_name}_actual']
        mae = errs.abs().mean()
        me = errs.mean()
        print(f'    {cfg_name:35s}  MAE={mae:6.2f}  me={me:+6.2f}')
    print()

## H/m subset

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm = valid[valid['target'].isin(HM)].copy()
print('H/m per-target (predicted / actual / err):\n')
print(f'{"target":32s} | ' + ' | '.join(f'{c[0]:24s}' for c in CONFIGS))
for _, r in hm.iterrows():
    cells_str = []
    for cfg_name, _, _ in CONFIGS:
        p = r[f'{cfg_name}_pred']
        a = r[f'{cfg_name}_actual']
        err = p - a
        cells_str.append(f'{p:6.1f}/{a:3.0f}/{err:+7.2f}'.ljust(24))
    print(f'{r["target"]:32s} | ' + ' | '.join(cells_str))

print()
print('H/m MAE aggregate:')
for cfg_name, _, _ in CONFIGS:
    errs = hm[f'{cfg_name}_pred'] - hm[f'{cfg_name}_actual']
    mae = errs.abs().mean()
    me = errs.mean()
    print(f'  {cfg_name:35s}  MAE={mae:6.2f}  me={me:+6.2f}')